<a href="https://colab.research.google.com/github/hhammza/Flyrank_ML_Internship_Hamza/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This notebook builds directly on my Week 1 lane: **Refresh / Content Opportunity Scoring** — ranking content pages by how urgently they need editorial review.

## 1. My lane as an ML task (type)

**Task type: Scoring, used to produce a ranking (with a binary classification proxy underneath).**

The end product editors need is not a single yes/no label per page — it's an *ordered list* they can work down from the top, because their review time is limited and they can only get to a fraction of pages this sprint. That output shape is a **ranking task**.

Under the hood, the score that produces the ranking is built from a **binary proxy signal** (is this page currently declining, yes/no — see Section 2), which is a classification-style problem. So the task is best described as: *classify/score each page's decline risk, then rank pages by that score.*

I ruled out the other options because:
- **Clustering** doesn't fit — I don't want to discover unlabeled groups of pages, I want to answer a specific question ("is this page worth reviewing soon?") that has a clear notion of higher/lower priority.
- **Pure classification** (declining vs. not) throws away information editors need. Two pages can both be "declining" but one has 50 impressions and the other has 50,000 — those aren't equally urgent, and a flat yes/no label collapses that difference. A score preserves it.

In [1]:
ml_task = {
    "task_type": "scoring -> ranking (binary decline-risk proxy underneath)",
    "why_not_clustering": "we have a specific target question, not an open discovery problem",
    "why_not_pure_classification": "flat yes/no collapses urgency differences between pages",
    "output_shape": "sorted list of pages, highest priority first",
}
print(ml_task)


{'task_type': 'scoring -> ranking (binary decline-risk proxy underneath)', 'why_not_clustering': 'we have a specific target question, not an open discovery problem', 'why_not_pure_classification': 'flat yes/no collapses urgency differences between pages', 'output_shape': 'sorted list of pages, highest priority first'}


## 2. Target or proxy

**This is a proxy label, not an observed outcome.**

There is no column in the dataset that records an editor's actual decision (e.g. "editor reviewed this page and confirmed it needed a refresh"). We don't have historical ground truth for "was this page correctly prioritized." So instead of a true label, I define a **rule-based proxy target**:

`needs_review_proxy = 1` if the page is (a) trending down **and** (b) has enough visibility (`impressions_90d >= 1000`) that a decline there is actually worth someone's time — else `0`.

For the ranking version, I'd extend this into a continuous **priority_score** that also factors in *how much* the page has declined and *how much* traffic is at stake, rather than a flat cutoff. But the binary version above is the simplest defensible proxy to start from.

**Why this matters (careful words, carried over from Week 1):** because this label is a rule I defined, not an outcome the world actually generated, any model trained on it is learning to imitate *my rule*, not to predict some independent ground truth. If the rule is wrong, the model will confidently be wrong in the same way. This should be validated with an actual content editor before being treated as truth.

In [13]:
# Proxy target definition — explicit and inspectable, not hidden in prose
proxy_target = {
    "name": "needs_review_proxy",
    "label_type": "proxy (rule-based), not an observed outcome",
    "rule": "trend_direction == 'down' AND impressions_90d >= 1000",
    "why_proxy_not_observed": "no column records an editor's actual review decision",
    "risk": "model will imitate the rule's blind spots, not ground truth",
    "mitigation": "validate the rule with a content editor before trusting it",
}
for k, v in proxy_target.items():
    print(f"{k}: {v}")

# Example of how the proxy would be computed once the data is loaded (see Section 4):
# df["needs_review_proxy"] = (
#     (df["trend_direction"] == "down") & (df["impressions_90d"] >= 1000)
# ).astype(int)
# print(df["needs_review_proxy"].value_counts(normalize=True))


name: needs_review_proxy
label_type: proxy (rule-based), not an observed outcome
rule: trend_direction == 'down' AND impressions_90d >= 1000
why_proxy_not_observed: no column records an editor's actual review decision
risk: model will imitate the rule's blind spots, not ground truth
mitigation: validate the rule with a content editor before trusting it


## 3. Success metric

**Metric: Precision@100** — of the top 100 pages my ranking puts at the top, what fraction actually match the `needs_review_proxy` label?

I'm choosing this over something like overall accuracy or AUC because of how the output is actually used: editors aren't going to review all 30,000 pages, they're going to work down a short list starting at the top. What matters is whether the *first* pages they see are worth their time — not how well the model does on pages nobody will ever look at. Precision@100 speaks directly to that: "if an editor only had time for 100 pages this sprint, how many of them would actually be worth reviewing?"

**What 'good' looks like:** roughly 54% of all pages are declining (from Week 1), so a random top-100 sample would already get ~54 hits by chance. I'd want Precision@100 meaningfully above that baseline — as a rough target, 80%+ — to justify using the model over random or manual triage.

In [14]:
success_metric = {
    "metric": "Precision@100",
    "definition": "share of the top-100 ranked pages that match needs_review_proxy == 1",
    "baseline_random": "~54% (overall declining-page share from Week 1 EDA)",
    "target": "80%+  (well above random, and within an editor's realistic sprint capacity)",
    "why_this_metric": "matches how the output is actually consumed - a short, worked-down list, not the full page set",
}
for k, v in success_metric.items():
    print(f"{k}: {v}")

# Once predictions exist, precision@100 would be computed like:
# top_100 = ranked_df.sort_values("priority_score", ascending=False).head(100)
# precision_at_100 = top_100["needs_review_proxy"].mean()
# print(f"Precision@100: {precision_at_100:.2%}")


metric: Precision@100
definition: share of the top-100 ranked pages that match needs_review_proxy == 1
baseline_random: ~54% (overall declining-page share from Week 1 EDA)
target: 80%+  (well above random, and within an editor's realistic sprint capacity)
why_this_metric: matches how the output is actually consumed - a short, worked-down list, not the full page set


## 4. The unit of analysis, as a real dataframe

One row = one pseudonymized content page. Loading the same starter dataset from Week 1 and showing it directly.

In [15]:
import pandas as pd

# Load the starter dataset (same one used in Week 1)
df = pd.read_csv("/content/content_refresh_anonymized.csv")

print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("One row = one content page\n")

# Show the unit of analysis directly
display(df.head(3))

print("\nColumns available for scoring:")
print(list(df.columns))

# Build the proxy target defined in Section 2, on the real data
df["needs_review_proxy"] = (
    (df["trend_direction"] == "down") & (df["impressions_90d"] >= 1000)
).astype(int)

print("\nProxy target distribution:")
print(df["needs_review_proxy"].value_counts(normalize=True).rename("share"))


Shape: 30,000 rows x 44 columns
One row = one content page



,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9



Columns available for scoring:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Proxy target distribution:
needs_review_proxy
0    0.7323
1    0.2677
Name: share, dtype: float64


## 5. Why ML beats a fixed rule here

A fixed if-statement rule (like the proxy in Section 2) is a fine *starting point*, but it breaks down as a *final* solution for a few reasons:

- **The signals interact, they don't just add up.** A page with a small traffic decline but very high impressions might deserve more attention than a page with a huge decline but almost no traffic — but *how* those two signals should trade off against each other isn't a single clean threshold, it's a curve, and probably a different curve depending on content type.
- **A flat threshold treats every page category the same.** `impressions_90d >= 1000` might be a meaningful cutoff for one type of page and meaningless for another (e.g. a niche technical page vs. a broad how-to page). Hand-tuning separate thresholds for every category doesn't scale across 44 features and thousands of pages.
- **There are ~44 features, and manually weighting all of them (and their interactions) by hand isn't realistic.** CTR, position, freshness, engagement, and trend all carry information, but nobody can hand-write an if-statement that correctly balances all of them at once — that's exactly the kind of pattern-combination problem ML is good at and manual rules aren't.
- **Rules don't adapt.** If page behavior shifts (seasonality, algorithm changes, a new content format), a fixed rule stays fixed until someone manually notices and rewrites it. A model can be retrained on fresh data.

In [16]:
why_ml_beats_rules = {
    "signal_interactions": "decline magnitude and visibility trade off nonlinearly, not additively",
    "threshold_inconsistency": "a single cutoff (e.g. 1000 impressions) doesn't generalize across page/content types",
    "feature_count": "~44 signals is too many to hand-weight reliably with if-statements",
    "adaptability": "rules go stale as page behavior shifts; models can be retrained",
}

for k, v in why_ml_beats_rules.items():
    print(f"{k}: {v}")


signal_interactions: decline magnitude and visibility trade off nonlinearly, not additively
threshold_inconsistency: a single cutoff (e.g. 1000 impressions) doesn't generalize across page/content types
feature_count: ~44 signals is too many to hand-weight reliably with if-statements
adaptability: rules go stale as page behavior shifts; models can be retrained


## Self-check

Before you submit, confirm each line honestly:

-  Every section above is filled — markdown thinking AND the code that backs it
-  The notebook runs top to bottom with no errors (Runtime → Run all)
-  No client names, URLs, or private queries anywhere
-  My claims use careful words: observed, measured, directional, decision-support
-  Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.